In [26]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
!pip install ucimlrepo

In [28]:
#importer les packages
import numpy as np
import pandas as pd
from scipy import stats
from ucimlrepo import fetch_ucirepo

# fetch dataset
regensburg_pediatric_appendicitis = fetch_ucirepo(id=938)

# data (as pandas dataframes)
X = regensburg_pediatric_appendicitis.data.features
y = regensburg_pediatric_appendicitis.data.targets

# metadata
print(regensburg_pediatric_appendicitis.metadata)
print(regensburg_pediatric_appendicitis.variables)


{'uci_id': 938, 'name': 'Regensburg Pediatric Appendicitis', 'repository_url': 'https://archive.ics.uci.edu/dataset/938/regensburg+pediatric+appendicitis', 'data_url': 'https://archive.ics.uci.edu/static/public/938/data.csv', 'abstract': 'This repository holds the data from a cohort of pediatric patients with suspected appendicitis admitted with abdominal pain to Children’s Hospital St. Hedwig in Regensburg, Germany, between 2016 and 2021. Each patient has (potentially multiple) ultrasound (US) images, aka views, tabular data comprising laboratory, physical examination, scoring results and ultrasonographic findings extracted manually by the experts, and three target variables, namely, diagnosis, management and severity.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Image'], 'num_instances': 782, 'num_features': 53, 'feature_types': ['Real', 'Categorical', 'Integer'], 'demographics': ['Age', 'Sex'], 'target_col': ['Management', 'Severity',

In [29]:
#ouvrir la base de données
df=pd.read_excel("/content/app_data.xlsx")
df
df.shape
df.info
df.describe()

,Age,BMI,Height,Weight,Length_of_Stay,Alvarado_Score,Paedriatic_Appendicitis_Score,Appendix_Diameter,Body_Temperature,WBC_Count,Neutrophil_Percentage,Segmented_Neutrophils,RBC_Count,Hemoglobin,RDW,Thrombocyte_Count,CRP,US_Number
count,781.000000,755.000000,756.000000,779.000000,778.000000,730.000000,730.000000,498.000000,775.000000,776.000000,679.000000,54.000000,764.000000,764.000000,756.000000,764.000000,771.000000,760.000000
mean,11.346483,18.906916,148.017460,43.172542,4.284062,5.921918,5.253425,7.762651,37.404516,12.670683,71.791163,64.929630,4.799490,13.380497,13.180291,285.252618,31.387899,425.515789
std,3.529979,4.385252,19.732016,17.390984,2.574057,2.155972,1.958456,2.536671,0.903678,5.366525,14.463656,15.085025,0.499012,1.393271,4.538774,72.494373,57.433588,271.585211
min,0.000000,7.827983,53.000000,3.960000,1.000000,0.000000,0.000000,2.700000,26.900000,2.600000,27.200000,32.000000,3.620000,8.200000,11.200000,91.000000,0.000000,1.000000
25%,9.200000,15.725294,137.000000,29.500000,3.000000,4.000000,4.000000,6.000000,36.800000,8.200000,61.400000,54.500000,4.537500,12.600000,12.300000,236.000000,1.000000,198.750000
50%,11.438741,18.062284,149.650000,41.400000,3.000000,6.000000,5.000000,7.500000,37.200000,12.000000,75.500000,64.500000,4.780000,13.300000,12.700000,276.000000,7.000000,398.500000
75%,14.099932,21.179011,163.000000,54.000000,5.000000,8.000000,7.000000,9.100000,37.900000,16.200000,83.600000,77.500000,5.020000,14.000000,13.300000,330.000000,33.000000,613.250000
max,18.360000,38.156221,192.000000,103.000000,28.000000,10.000000,10.000000,17.000000,40.200000,37.700000,97.700000,91.000000,14.000000,36.000000,86.900000,708.000000,365.000000,992.000000


In [30]:
#supprimer les lignes repetées
df.duplicated().sum()


np.int64(0)

In [31]:
#découverte des valeurs manquantes
df.isnull().sum()

,0
Age,1
BMI,27
Sex,2
Height,26
Weight,3
Length_of_Stay,4
Management,1
Severity,1
Diagnosis_Presumptive,2
Diagnosis,2


In [32]:
#gestion des valeurs manquantes

  #gestion des variables manquantes de type faible et modéré

      #gestion des variables de type numérique
colonnes_continues = [
    'Appendix_Diameter', 'Neutrophil_Percentage', 'Alvarado_Score',
    'Paedriatic_Appendicitis_Score', 'BMI', 'Height', 'RDW',
    'US_Number', 'Hemoglobin', 'Thrombocyte_Count', 'RBC_Count',
    'CRP', 'Body_Temperature', 'WBC_Count', 'Length_of_Stay',
    'Weight', 'Age'
]

for col in colonnes_continues:
    df[col] = df[col].fillna(df[col].median())

     #gestion des variables catégorielles
colonnes_cat = ['RBC_in_Urine','Ketones_in_Urine','WBC_in_Urine','Ipsilateral_Rebound_Tenderness'
    'Sex', 'Diagnosis', 'Severity', 'Management', 'Neutrophilia',
    'Migratory_Pain', 'Lower_Right_Abd_Pain', 'Contralateral_Rebound_Tenderness',
    'Coughing_Pain', 'Nausea', 'Loss_of_Appetite', 'Dysuria',
    'Stool', 'Peritonitis', 'Psoas_Sign', 'Appendix_on_US',
    'US_Performed', 'Free_Fluids','Diagnosis_Presumptive']

for col in colonnes_cat:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

In [33]:
#gestion des variables de type critique
colonnes_critiques = [
    'Abscess_Location', 'Gynecological_Findings', 'Conglomerate_of_Bowel_Loops',
    'Segmented_Neutrophils', 'Ileus', 'Perfusion', 'Enteritis', 'Appendicolith',
    'Coprostasis', 'Perforation', 'Appendicular_Abscess', 'Bowel_Wall_Thickening',
    'Lymph_Nodes_Location', 'Target_Sign', 'Meteorism', 'Pathological_Lymph_Nodes',
    'Appendix_Wall_Layers', 'Surrounding_Tissue_Reaction']

df = df.drop(columns=colonnes_critiques)

In [34]:
#gestion des valeurs aberrantes
   #Détection par IQR
colonnes_numeriques = [
    'Age', 'BMI', 'CRP', 'WBC_Count', 'Hemoglobin',
    'Body_Temperature', 'RDW', 'Appendix_Diameter',
    'Alvarado_Score', 'Paedriatic_Appendicitis_Score',
    'Length_of_Stay', 'Weight', 'Height', 'RBC_Count',
    'Thrombocyte_Count', 'Neutrophil_Percentage'
]

for col in colonnes_numeriques:
    s = df[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    bb, bh = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n = ((s < bb) | (s > bh)).sum()
    print(f'{col}: {n} outliers | bornes=[{bb:.2f}, {bh:.2f}]')

 #suppression des valeurs biologiques impossibles
df = df[df['Body_Temperature'] > 34]
df = df[df['Hemoglobin'] <= 20]
df = df[df['RDW'] <= 30]
df = df[df['RBC_Count'] <= 8]

print(f'Lignes restantes : {len(df)}')

 #Winnorisation
colonnes_winsor = ['BMI', 'WBC_Count', 'Length_of_Stay',
                   'Thrombocyte_Count', 'Appendix_Diameter']

for col in colonnes_winsor:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=borne_basse, upper=borne_haute)


Age: 5 outliers | bornes=[1.90, 21.39]
BMI: 30 outliers | bornes=[7.99, 28.83]
CRP: 91 outliers | bornes=[-45.50, 78.50]
WBC_Count: 7 outliers | bornes=[-3.55, 28.05]
Hemoglobin: 26 outliers | bornes=[10.75, 15.95]
Body_Temperature: 12 outliers | bornes=[35.15, 39.55]
RDW: 23 outliers | bornes=[10.80, 14.80]
Appendix_Diameter: 217 outliers | bornes=[5.50, 9.50]
Alvarado_Score: 0 outliers | bornes=[-2.00, 14.00]
Paedriatic_Appendicitis_Score: 14 outliers | bornes=[1.00, 9.00]
Length_of_Stay: 44 outliers | bornes=[0.00, 8.00]
Weight: 8 outliers | bornes=[-7.25, 90.75]
Height: 15 outliers | bornes=[101.66, 198.56]
RBC_Count: 15 outliers | bornes=[3.84, 5.71]
Thrombocyte_Count: 9 outliers | bornes=[96.88, 467.88]
Neutrophil_Percentage: 5 outliers | bornes=[36.00, 110.20]
Lignes restantes : 776


In [35]:
colonnes_a_supprimer = list(set([
    'Peritonitis', 'Weight', 'Height', 'Alvarado_Score', 'Neutrophilia', 'Stool', 'RBC_Count', 'Paedriatic_Appendicitis_Score',
    'Management', 'Severity', 'Length_of_Stay', 'Diagnosis_Presumptive', 'US_Number', 'CRP_log'
]))
df = df.drop(columns=[col for col in colonnes_a_supprimer if col in df.columns])
print(df.shape)
print(df.columns)

(776, 27)
Index(['Age', 'BMI', 'Sex', 'Diagnosis', 'Appendix_on_US', 'Appendix_Diameter',
       'Migratory_Pain', 'Lower_Right_Abd_Pain',
       'Contralateral_Rebound_Tenderness', 'Coughing_Pain', 'Nausea',
       'Loss_of_Appetite', 'Body_Temperature', 'WBC_Count',
       'Neutrophil_Percentage', 'Hemoglobin', 'RDW', 'Thrombocyte_Count',
       'Ketones_in_Urine', 'RBC_in_Urine', 'WBC_in_Urine', 'CRP', 'Dysuria',
       'Psoas_Sign', 'Ipsilateral_Rebound_Tenderness', 'US_Performed',
       'Free_Fluids'],
      dtype='object')


In [ ]:
df.to_excel('app_data_sans_critiques.xlsx', index=False)
print(f'Fichier sauvegardé : {df.shape[0]} lignes, {df.shape[1]} colonnes')

#test